<a href="https://colab.research.google.com/github/Harman1199/abb/blob/main/abb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code

In [ ]:
%%capture
try:
    import xlsxwriter
except ImportError:
    %pip install xlsxwriter

In [ ]:
import os
import sys

from datetime import datetime, timedelta
today = datetime.today().date()

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Helper Functions

def remove(criteria=""):
    for input_file in os.listdir():
        if input_file.endswith(".xlsx") and criteria in input_file:
            os.remove(input_file)
            print(f"Deleted {input_file}")

def warn(suppress_warnings, message):
    print(f"\n\nWARNING: {message}")
    if not suppress_warnings:
        sys.exit("\n\nProgram execution stopped.\nSet suppress_warnings to True to ignore the warning and continue anyway.")

def parse_date(abb_data, price_col):
    price_col_date_range = abb_data[price_col - 1][abb_data[abb_data[price_col - 1].astype(str).str.contains("Final Net Price")].index.item() + 1]
    start_date = pd.to_datetime(price_col_date_range.split(" ")[0].strip("("), format="%m/%d/%Y").date()
    end_date = pd.to_datetime(price_col_date_range.split(" ")[2].strip(")"), format="%m/%d/%Y").date()

    return start_date, end_date

def load_input(dir_path=os.listdir()):
    c_list = {}
    for input_file in dir_path:
        if input_file.endswith(".xlsx") and "changes" not in input_file and "pages" not in input_file:
            if "(1)" not in input_file:
                contract = input_file.split("-")[1] # name
                c_list[contract] = input_file

    c_list = dict(sorted(c_list.items(), key=lambda item: int(item[0])))
    return c_list.keys(), c_list

In [ ]:
# to ensure dates are ok and pages are active
def check_pages(contract, curr_data):
    curr_data_func = curr_data[['price_page_uid', 'item_id', 'effective_date', 'expiration_date', 'row_status_flag']]
    curr_data_func['effective_date'] = pd.to_datetime(curr_data_func['effective_date'].astype(str).str.strip(), format="%m/%d/%Y")
    curr_data_func['expiration_date'] = pd.to_datetime(curr_data_func['expiration_date'].astype(str).str.strip(), format="%m/%d/%Y")

    if (curr_data_func['row_status_flag'] != "active").any():
        raise AssertionError(contract, "Deleted price page(s) present")
    if (curr_data_func['effective_date'] > datetime.today()).any():
        raise AssertionError(contract, "Page(s) with future effective date")
    if (curr_data_func['expiration_date'] <= datetime.today()).any():
        raise AssertionError(contract, "Expired price page(s)")
    if curr_data_func['expiration_date'].nunique() > 1:
        raise AssertionError(contract, "Pages with multiple expiry dates")
    if curr_data_func['item_id'].duplicated().any():
        raise AssertionError(contract, "Duplicate item page(s)")

In [ ]:
# to extract customer data from ABB file
def get_customer_list(df):
    abb_customers = df[(df[1].notna()) & (df[8].isna())]
    abb_customers = abb_customers[[1, 2]].reset_index(drop=True)
    abb_customers.columns = ['customer', 'account_no']
    # converting 'account_no' to numeric, coercing errors to NaN
    abb_customers['account_no'] = pd.to_numeric(abb_customers['account_no'], errors='coerce')
    # dropping rows where 'account_no' could not be converted to a number
    abb_customers.dropna(subset=['account_no'], inplace=True)
    # abb_customers['account_no'] = abb_customers['account_no'].astype(int)
    return abb_customers['customer']

In [ ]:
def assess_dates(contract, abb_data, price_col, suppress_warnings):
    if not abb_data.loc[abb_data[9].notna(), :][9].astype(str).str.contains("CAD").all():
        print("Error: Could not read data from ABB file as structure has changed.")
        sys.exit()

    # creating abb_rows to check other CAD cols
    abb_rows = abb_data.loc[abb_data[9].notna(), :]
    cad_cols = [col for col in abb_rows.columns if abb_rows[col].astype(str).str.contains("CAD").all()]

    if len(cad_cols) > 1:
        print(f"{len(cad_cols) - 1} Future column(s) with start date(s): ", end="")
        f_cols = cad_cols[1:].copy()

        for idx, col in enumerate(f_cols):
            start_date, _ = parse_date(abb_data, col - 1)
            if idx == len(f_cols) - 1:
                print(start_date)
            else:
                print(start_date, end=", ")

    if price_col > 8 and len(cad_cols) <= 1:
        print("No future columns exist. Comparing against current ABB data")
        price_col = 8

    start_date, end_date = parse_date(abb_data, price_col)
    if (price_col == 8) and not (start_date <= today <= end_date):
        warn(suppress_warnings, "Contract not valid. Looking at 'Current' spa costs column")
    if (price_col == 11) and not (today < start_date):
        warn(suppress_warnings, "Contract already started. Looking at 'Future' spa costs column")

    return price_col

In [ ]:
def get_changes_df(curr, contract, column):
    # changes = curr_mismatched_rows[['item_id', 'other_cost_value', f'ABB_{column}']]
    changes = curr[['price_page_uid', 'expiration_date', 'item_id', 'Description', 'Price UOM', 'other_cost_value', f'ABB_{column}']]
    changes.columns = ['Page ID', 'Expiration Date', 'Item ID', 'Description', 'Price UOM', 'Current SPA Cost', f'ABB_{column}']
    changes['Contract'] = int(contract) if isinstance(contract, int) else contract
    changes = changes[['Contract', 'Page ID', 'Expiration Date', 'Item ID', 'Description', 'Price UOM', 'Current SPA Cost', f'ABB_{column}']]

    changes['%Change'] = round(((changes[f'ABB_{column}'] - changes['Current SPA Cost']) / changes['Current SPA Cost']) * 100, 2)
    changes['Status'] = "Modified"

    changes.loc[changes['%Change'] == 0, '%Change'] = ""
    changes.loc[changes['%Change'] == "", 'Status'] = ""

    changes.loc[changes['Page ID'] == 0, 'Status'] = "New"
    changes.loc[changes[f'ABB_{column}'].isna(), 'Status'] = "Removed"
    return changes

In [ ]:
# Generate Changes file
def create_change_file(changes, contract):
    changes = changes.drop(columns=['Expiration Date'])

    with pd.ExcelWriter(f"changes_{contract}.xlsx", engine='xlsxwriter') as writer:
        changes.to_excel(writer, sheet_name='changes', index=False)

        wb_changes = writer.book
        ws_changes = writer.sheets['changes']

        green_format = wb_changes.add_format({'bg_color': '#C6EFCE'})
        red_format = wb_changes.add_format({'bg_color': '#FFC7CE'})

        percentage_col_index = changes.columns.get_loc('%Change')
        status_col_index = changes.columns.get_loc('Status')

        # color increases as red
        ws_changes.conditional_format(1, percentage_col_index, len(changes), percentage_col_index, {
            'type': 'cell',
            'criteria': '>',
            'value': 0,
            'format': red_format
        })
        # color decreases as green
        ws_changes.conditional_format(1, percentage_col_index, len(changes), percentage_col_index, {
            'type': 'cell',
            'criteria': '<',
            'value': 0,
            'format': green_format
        })

        end_row = len(changes.columns) - 1
        for row_num, row in changes.iterrows():
            if row['Status'] == "New":
                ws_changes.conditional_format(row_num + 1, 0, row_num + 1, end_row, {
                    'type': 'no_blanks',
                    'format': green_format
                })
            if row['Status'] == "Removed":
                ws_changes.conditional_format(row_num + 1, 0, row_num + 1, end_row, {
                    'type': 'no_blanks',
                    'format': red_format
                })

        ws_changes.autofit()
    print(f"\nChanges file generated: changes_{contract}.xlsx")

In [ ]:
# Generate Mass Update file
def create_mass_update_file(changes, contract, column):
    changes = changes[['Page ID', 'Expiration Date', 'Item ID', 'Current SPA Cost', f'ABB_{column}', 'Status']]
    had_new = len(changes[changes['Status'] == "New"])
    changes = changes[changes['Status'] != "New"] # !handle new pages separately
    changes.reset_index(drop=True, inplace=True)

    changes[f'ABB_{column}'] = changes[f'ABB_{column}'].fillna(changes['Current SPA Cost'])

    col_list_pricing = ['ID', 'Price Page ID', 'Expiration Date', 'Status', 'Item', 'Save Changes']
    col_list_costs = ['ID', 'Main ID', 'Price Page ID', 'Other Cost Type Cd', 'Other Cost Value',
                    #   'Secondary Rebate Type Cd', 'Secondary Rebate Calc Meth Cd', 'Secondary Rebate Source Cd',
                    #   'Commission Cost Type Cd', 'Commission Cost Source Cd', 'Commission Cost Calc Method Cd', 'Commission Cost Calc Value',
                      'Save Changes']


    pricing_tab = pd.DataFrame(columns=col_list_pricing)
    costs_tab = pd.DataFrame(columns=col_list_costs)

    pricing_tab['Price Page ID'] = changes['Page ID'].astype(int)
    pricing_tab['Expiration Date'] = changes['Expiration Date']
    # Delete pages of items which are Removed
    pricing_tab['Status'] = changes['Status'].apply(lambda x: "Delete" if x == "Removed" else "Active")
    # Set Expiration Date of Deleted pages as Today's date
    pricing_tab.loc[pricing_tab['Status'] == "Delete", 'Expiration Date'] = today.strftime("%m/%d/%Y 00:00:00")
    pricing_tab['Item'] = changes['Item ID']

    pricing_tab['ID'] = np.arange(1, len(pricing_tab) + 1)
    pricing_tab['Save Changes'] = 'Y'


    costs_tab['ID'] = pricing_tab['ID']
    costs_tab['Main ID'] = pricing_tab['ID']
    costs_tab['Price Page ID'] = pricing_tab['Price Page ID']
    costs_tab['Other Cost Type Cd'] = "Value"
    costs_tab['Other Cost Value'] = changes[f'ABB_{column}']


    # addition
    # costs_tab['Secondary Rebate Type Cd'] = "Source"
    # costs_tab['Secondary Rebate Calc Meth Cd'] = "Subtract"
    # costs_tab['Secondary Rebate Source Cd'] = "Primary Supplier Cost"
    # costs_tab['Commission Cost Type Cd'] = "Source"
    # costs_tab['Commission Cost Source Cd'] = "Other Cost"
    # costs_tab['Commission Cost Calc Method Cd'] = "Multiplier"
    # costs_tab['Commission Cost Calc Value'] = 1
    ###

    costs_tab['Save Changes'] = "Y"


    with pd.ExcelWriter(f"pages_{contract}.xlsx") as writer:
        pricing_tab.to_excel(writer, sheet_name="Sales Pricing", index=False)
        costs_tab.to_excel(writer, sheet_name="Costs", index=False)

        wb_pages = writer.book
        ws_sales_pricing = writer.sheets['Sales Pricing']
        ws_costs = writer.sheets['Costs']

        red_format = wb_pages.add_format({'bg_color': '#FFC7CE'})
        status_col_index = changes.columns.get_loc('Status')

        end_row = len(pricing_tab.columns) - 1
        for row_num, row in pricing_tab.iterrows():
            if row['Status'] == "Delete":
                ws_sales_pricing.conditional_format(row_num + 1, 0, row_num + 1, end_row, {
                    'type': 'no_blanks',
                    'format': red_format
                })

        ws_sales_pricing.autofit()
        ws_costs.autofit()

    print(f"\n\nMass update file generated: pages_{contract}.xlsx")
    if had_new:
        print(f"** Please manually take care of {had_new} new page(s). **")

In [ ]:
# Main function
def verify(column="Current", generate_mass_update_file=False, generate_change_file=False, drive=False, suppress_warnings=False):
    try:
        num_results, c_list = load_input(dir_path=os.listdir())
        print(f"Showing results from {len(num_results)} contract(s)\n")

        if not c_list:
            print("No files found")
            return

        # mount Drive if not found in /content
        if drive and "drive" not in os.listdir():
            from google.colab import drive
            from contextlib import redirect_stdout, redirect_stderr

            with open(os.devnull, 'w') as f:
                with redirect_stdout(f), redirect_stderr(f):
                    drive.mount('/content/drive')

        col_num_name_map = {
            8: "Current",
            11: "Future"
        }

        if column == "Current":
            price_col = 8
        elif column == "Future":
            price_col = 11
        else:
            print("Invalid column value. Please type 'Current' or 'Future'")
            return

        for contract in c_list:
            print(f"{contract}:")
            workbook = pd.ExcelFile(c_list[contract])
            try:
                if drive:
                    curr_data = pd.read_excel(f'/content/drive/MyDrive/contract_states/{contract}.xlsx', sheet_name='LIST.list_detail')
                else:
                    # trying to read p21 page data from abb file
                    curr_data = pd.read_excel(workbook, sheet_name='LIST.list_detail')
            except Exception as e:
                print(f"Error: {contract} - P21 page data not available or could not be loaded")
                return

            # checking page validity
            check_pages(contract, curr_data)

            curr_data = curr_data[['item_id', 'other_cost_value', 'price_page_uid', 'expiration_date']]
            curr_data['price_page_uid'] = curr_data['price_page_uid'].astype(int)

            abb_data = pd.read_excel(workbook, sheet_name='Pricing-Report', header=None)


            # only verifying customer data if drive set to True and file available in drive
            # verifying customers ##########################################
            if drive:
                try:
                    curr_customers = pd.read_excel('/content/drive/MyDrive/abb_customers.xlsx')[['contract', 'customer']]
                except Exception as e:
                    print("Error: Could not read previous customer data")
                    return
                abb_customers = get_customer_list(abb_data)

                for file_ in os.listdir():
                    if file_ == (c_list[contract].split(".")[0] + " (1).xlsx"):
                        other_loc_workbook = pd.ExcelFile(file_)
                        other_loc_workbook = pd.read_excel(other_loc_workbook, sheet_name='Pricing-Report', header=None)
                        other_loc_customers = get_customer_list(other_loc_workbook)
                        abb_customers = pd.concat([abb_customers, other_loc_customers], ignore_index=True)

                if set(curr_customers[curr_customers['contract'] == int(contract)]['customer']) == set(abb_customers):
                    print(f"All customers exist ({len(set(abb_customers))})")
                else:
                    print("Customers missing or extra")
                    print(set(curr_customers[curr_customers['contract'] == int(contract)]['customer']).symmetric_difference(set(abb_customers)))
            ################################################################

            # assessing abb dates
            price_col = assess_dates(contract, abb_data, price_col, suppress_warnings)

            # ABB data manipulation
            abb_data = abb_data[abb_data[1].notna()].iloc[:, :16]
            abb_rows = abb_data.loc[abb_data[price_col].notna(), :]
            abb_rows = abb_rows[[1, 2, 5, price_col]]
            abb_rows[0] = "ABB-" + abb_rows[1]
            abb_rows = abb_rows[[0, 1, 2, 5, price_col]]
            abb_rows.columns = ['item_id', 'catalog', 'Description', 'Price UOM', f'ABB_{column}']
            abb_rows[f'ABB_{column}'] = abb_rows[f'ABB_{column}'].astype(float)
            abb_rows.reset_index(drop=True, inplace=True)
            ################################################################

            if set(curr_data['item_id']) == set(abb_rows['item_id']):
                print("All items exist")
            else:
                print("Items missing or extra")

            # stats
            print(f"P21: {len(curr_data)} | Matched: {curr_data['item_id'].isin(abb_rows['item_id']).sum()} | ABB: {len(abb_rows)}")

            curr = pd.merge(abb_rows, curr_data, on='item_id',  how='outer')
            # filling NaN with 0 before converting to int
            curr['price_page_uid'] = curr['price_page_uid'].fillna(0).astype(int)

            # displaying comparison and file generation
            curr_mismatched_rows = curr[curr[f'ABB_{column}'] != curr['other_cost_value']]
            if len(curr_mismatched_rows):
                print(f"\n{column} ({len(curr_mismatched_rows)} mismatches)")
                print()
                curr_mismatched_rows.rename(columns={
                    'other_cost_value': 'current_SPA_cost',
                    }, inplace=True)
                display(curr_mismatched_rows[[ 'price_page_uid', 'item_id', 'current_SPA_cost', f'ABB_{column}']])

                # File Generation
                changes = get_changes_df(curr, contract, column)
                if generate_change_file:
                    create_change_file(changes, contract)
                if generate_mass_update_file:
                    create_mass_update_file(changes, contract, column)
            else:
                print(f"{col_num_name_map[price_col]}...OK")

            print("#" * 50)
            print("\n")
    except SystemExit as e:
        print(e)

# Result

In [ ]:
# remove()

In [ ]:
verify(

    ## Configuration #######
    # (True / False)
    drive=False,
    suppress_warnings=False,

    # ("Current" / "Future")
    column="Current",
    ########################


    ## Files #######################
    # (True / False)
    generate_change_file=True,
    generate_mass_update_file=True,
    ################################
)